# Gerador de Casos Sintéticos — Clinical Reasoning

Objetivo:
- gerar casos clínicos mais realistas e úteis para treino
- reduzir viés para condições graves sem evidência suficiente
- incluir casos comuns, negativos explícitos, dados insuficientes e fora de escopo


In [ ]:
import json
import random
from collections import Counter

SEED = 42
random.seed(SEED)

SEXOS = ["masculino", "feminino"]
COMORBIDADES_POOL = [
    "hipertensão arterial", "diabetes tipo 2", "insuficiência cardíaca",
    "DPOC", "doença renal crônica", "hipotireoidismo",
    "obesidade", "dislipidemia", "asma", "tabagismo"
]

QUESTIONS = [
    "Qual hipótese diagnóstica é mais provável?",
    "Quais diagnósticos devem ser considerados?",
    "Quais exames auxiliariam na investigação inicial?",
    "Como priorizar as hipóteses neste caso?",
]

SUPPORTED_CASES = {
    "dengue": {
        "positives": ["febre alta", "cefaleia intensa", "mialgia", "dor retro-orbitária"],
        "negatives": ["sem rigidez de nuca", "sem fotofobia", "sem alteração de consciência"],
        "differentials": ["virose inespecífica", "meningite bacteriana", "sinusite aguda"],
        "exams": ["hemograma", "hematócrito", "sorologia ou teste rápido conforme contexto epidemiológico"],
        "reasoning_for": [
            "Febre alta associada a cefaleia e dor retro-orbitária favorece arbovirose, especialmente dengue.",
            "O padrão clínico é mais compatível com síndrome febril viral do que com quadro meníngeo clássico."
        ],
        "reasoning_against": [
            "A ausência de rigidez de nuca, fotofobia e alteração de consciência reduz a probabilidade de meningite.",
        ],
        "comorbidity_weight": 0.25,
    },
    "virose inespecífica": {
        "positives": ["febre baixa", "mal-estar", "coriza", "odinofagia leve"],
        "negatives": ["sem dispneia", "sem toxemia", "sem rigidez de nuca"],
        "differentials": ["IVAS", "dengue", "pneumonia"],
        "exams": ["avaliação clínica", "testes adicionais apenas se houver piora ou red flags"],
        "reasoning_for": [
            "O quadro é compatível com síndrome viral leve de vias aéreas superiores.",
        ],
        "reasoning_against": [
            "A ausência de dispneia e de toxemia reduz a probabilidade de pneumonia ou sepse.",
        ],
        "comorbidity_weight": 0.15,
    },
    "sinusite aguda": {
        "positives": ["dor facial", "congestão nasal", "cefaleia frontal", "secreção nasal purulenta"],
        "negatives": ["sem rigidez de nuca", "sem alteração de consciência", "sem déficit neurológico focal"],
        "differentials": ["IVAS", "meningite bacteriana", "enxaqueca"],
        "exams": ["avaliação clínica", "imagem apenas em casos complicados ou atípicos"],
        "reasoning_for": [
            "Dor facial com congestão nasal e secreção purulenta favorece sinusite aguda.",
        ],
        "reasoning_against": [
            "A ausência de sinais meníngeos e neurológicos reduz a chance de meningite ou evento neurológico agudo.",
        ],
        "comorbidity_weight": 0.15,
    },
    "enxaqueca": {
        "positives": ["cefaleia pulsátil unilateral", "fotofobia", "fonofobia", "náuseas"],
        "negatives": ["sem rigidez de nuca", "sem alteração de consciência", "sem déficit neurológico focal"],
        "differentials": ["cefaleia tensional", "sinusite aguda", "hemorragia subaracnoide"],
        "exams": ["avaliação clínica", "neuroimagem apenas se houver red flags"],
        "reasoning_for": [
            "Cefaleia pulsátil unilateral com fotofobia, fonofobia e náuseas é altamente compatível com enxaqueca.",
        ],
        "reasoning_against": [
            "A ausência de thunderclap, alteração de consciência e sinais focais reduz a chance de hemorragia subaracnoide.",
        ],
        "comorbidity_weight": 0.10,
    },
    "cefaleia tensional": {
        "positives": ["cefaleia em banda", "dor leve a moderada", "cefaleia bilateral"],
        "negatives": ["sem náuseas", "sem fotofobia importante", "sem alteração de consciência"],
        "differentials": ["enxaqueca", "sinusite aguda", "hemorragia subaracnoide"],
        "exams": ["avaliação clínica", "investigação adicional apenas se surgirem red flags"],
        "reasoning_for": [
            "Cefaleia em banda, bilateral e sem sintomas autonômicos ou neurológicos favorece cefaleia tensional.",
        ],
        "reasoning_against": [
            "A ausência de náuseas importantes, fotofobia marcante e thunderclap reduz a probabilidade de enxaqueca ou HSA.",
        ],
        "comorbidity_weight": 0.05,
    },
    "IVAS": {
        "positives": ["dor de garganta", "febre baixa", "tosse seca", "linfonodos cervicais dolorosos"],
        "negatives": ["sem dispneia", "sem toxemia", "sem rigidez de nuca"],
        "differentials": ["virose inespecífica", "sinusite aguda", "pneumonia"],
        "exams": ["avaliação clínica", "teste rápido conforme contexto"],
        "reasoning_for": [
            "A combinação de odinofagia, febre baixa e sintomas de vias aéreas superiores favorece IVAS.",
        ],
        "reasoning_against": [
            "Sem dispneia ou toxemia, pneumonia e sepse ficam menos prováveis.",
        ],
        "comorbidity_weight": 0.10,
    },
    "refluxo gastroesofágico": {
        "positives": ["queimação retroesternal pós-prandial", "piora ao deitar", "regurgitação ácida"],
        "negatives": ["sem sudorese", "sem irradiação típica", "sem relação clara com esforço"],
        "differentials": ["síndrome coronariana aguda", "ansiedade", "dor torácica inespecífica"],
        "exams": ["avaliação clínica", "ECG se perfil de risco justificar", "teste terapêutico conforme contexto"],
        "reasoning_for": [
            "Relação com refeições e piora ao deitar favorecem refluxo gastroesofágico.",
        ],
        "reasoning_against": [
            "A ausência de padrão típico isquêmico reduz a probabilidade de síndrome coronariana aguda, embora não a exclua em pacientes de risco.",
        ],
        "comorbidity_weight": 0.10,
    },
    "crise de ansiedade": {
        "positives": ["palpitações", "hiperventilação", "parestesias", "dor torácica em pontada"],
        "negatives": ["sem síncope", "sem dispneia súbita importante", "sem dor opressiva típica"],
        "differentials": ["síndrome coronariana aguda", "tromboembolismo pulmonar", "refluxo gastroesofágico"],
        "exams": ["avaliação clínica", "ECG se primeiro episódio ou dúvida diagnóstica"],
        "reasoning_for": [
            "Palpitações, hiperventilação e parestesias em conjunto favorecem crise de ansiedade ou síndrome de hiperventilação.",
        ],
        "reasoning_against": [
            "A ausência de dor opressiva típica, síncope e sinais de instabilidade reduz a probabilidade de causa cardiopulmonar grave.",
        ],
        "comorbidity_weight": 0.05,
    },
    "lombalgia mecânica": {
        "positives": ["dor lombar após esforço físico", "piora ao movimento", "melhora parcial com repouso"],
        "negatives": ["sem febre", "sem disúria", "sem déficit neurológico"],
        "differentials": ["cólica renal", "pielonefrite", "radiculopatia"],
        "exams": ["avaliação clínica", "imagem apenas se houver red flags"],
        "reasoning_for": [
            "Dor lombar relacionada a esforço e sem sinais sistêmicos favorece lombalgia mecânica.",
        ],
        "reasoning_against": [
            "A ausência de febre, disúria e sinais neurológicos reduz causas infecciosas ou neurológicas graves.",
        ],
        "comorbidity_weight": 0.10,
    },
    "otite média aguda": {
        "positives": ["otalgia", "febre", "otorreia", "hipoacusia"],
        "negatives": ["sem rigidez de nuca", "sem alteração de consciência"],
        "differentials": ["otite externa", "mastoidite", "IVAS"],
        "exams": ["otoscopia", "avaliação clínica"],
        "reasoning_for": [
            "Otalgia associada a febre e hipoacusia favorece otite média aguda.",
        ],
        "reasoning_against": [
            "Sem sinais de complicação intracraniana, diagnósticos neurológicos graves ficam menos prováveis.",
        ],
        "comorbidity_weight": 0.10,
    },
    "herpes zoster": {
        "positives": ["dor neuropática em faixa", "hiperestesia", "distribuição unilateral"],
        "negatives": ["sem dor opressiva típica", "sem dispneia", "sem trauma local"],
        "differentials": ["dor neuropática inespecífica", "dor musculoesquelética", "síndrome coronariana aguda"],
        "exams": ["avaliação clínica", "reavaliação em 24-72h se exantema ainda ausente"],
        "reasoning_for": [
            "Dor neuropática unilateral em faixa dermatomérica favorece herpes zoster, mesmo antes da erupção cutânea.",
        ],
        "reasoning_against": [
            "O padrão neuropático localizado torna menos provável causa cardiopulmonar aguda clássica.",
        ],
        "comorbidity_weight": 0.10,
    },
    "hipotireoidismo": {
        "positives": ["cansaço", "intolerância ao frio", "constipação", "bradicardia"],
        "negatives": ["sem febre", "sem dor aguda", "sem sinais focais"],
        "differentials": ["insuficiência cardíaca descompensada", "anemia", "depressão"],
        "exams": ["TSH", "T4 livre", "hemograma"],
        "reasoning_for": [
            "Fadiga, intolerância ao frio, constipação e bradicardia favorecem hipotireoidismo.",
        ],
        "reasoning_against": [
            "A evolução arrastada e a ausência de síndrome febril ou dor aguda reduzem causas infecciosas e eventos agudos.",
        ],
        "comorbidity_weight": 0.20,
    },
    "pielonefrite": {
        "positives": ["febre alta", "dor lombar", "disúria", "polaciúria"],
        "negatives": ["sem dor pleurítica", "sem rigidez de nuca"],
        "differentials": ["cólica renal", "cistite", "sepse"],
        "exams": ["urina tipo I", "urocultura", "hemograma"],
        "reasoning_for": [
            "Febre alta associada a dor lombar e sintomas urinários favorece pielonefrite.",
        ],
        "reasoning_against": [
            "O conjunto é mais compatível com foco urinário alto do que com causas torácicas ou neurológicas.",
        ],
        "comorbidity_weight": 0.20,
    },
    "cólica renal": {
        "positives": ["dor lombar intensa irradiando para a virilha", "hematúria", "náuseas", "inquietação"],
        "negatives": ["sem febre", "sem disúria importante", "sem toxemia"],
        "differentials": ["pielonefrite", "lombalgia mecânica", "apendicite aguda"],
        "exams": ["urina tipo I", "ultrassom renal", "tomografia sem contraste conforme contexto"],
        "reasoning_for": [
            "Dor lombar intensa irradiando para virilha com hematúria favorece cólica renal.",
        ],
        "reasoning_against": [
            "A ausência de febre e toxemia reduz foco infeccioso alto como pielonefrite.",
        ],
        "comorbidity_weight": 0.10,
    },
    "gastroenterite aguda": {
        "positives": ["dor abdominal difusa", "diarreia", "náuseas", "vômitos"],
        "negatives": ["sem dor localizada em quadrante inferior direito", "sem sinais peritoneais", "sem rigidez de nuca"],
        "differentials": ["apendicite aguda", "intoxicação alimentar", "virose inespecífica"],
        "exams": ["avaliação clínica", "hidratação", "hemograma se sinais de gravidade"],
        "reasoning_for": [
            "Dor abdominal difusa com diarreia e náuseas favorece gastroenterite aguda.",
        ],
        "reasoning_against": [
            "Sem localização em QID ou sinais peritoneais, apendicite fica menos provável.",
        ],
        "comorbidity_weight": 0.10,
    },
    "apendicite aguda": {
        "positives": ["dor em quadrante inferior direito", "náuseas", "vômitos", "febre baixa"],
        "negatives": ["sem diarreia importante", "sem sintomas respiratórios"],
        "differentials": ["gastroenterite aguda", "cólica renal", "doença inflamatória pélvica"],
        "exams": ["hemograma", "PCR", "ultrassom abdominal ou tomografia"],
        "reasoning_for": [
            "Dor localizada em quadrante inferior direito associada a náuseas e febre baixa favorece apendicite aguda.",
        ],
        "reasoning_against": [
            "A ausência de diarreia importante reduz gastroenterite como hipótese principal.",
        ],
        "comorbidity_weight": 0.10,
    },
    "síndrome coronariana aguda": {
        "positives": ["dor torácica opressiva", "sudorese", "dispneia", "náuseas"],
        "negatives": ["sem caráter pleurítico predominante", "sem relação com posição", "sem reprodução à palpação"],
        "differentials": ["tromboembolismo pulmonar", "refluxo gastroesofágico", "insuficiência cardíaca descompensada"],
        "exams": ["ECG", "troponina", "ecocardiograma"],
        "reasoning_for": [
            "Dor torácica opressiva com sudorese e dispneia favorece síndrome coronariana aguda.",
        ],
        "reasoning_against": [
            "A ausência de caráter pleurítico predominante torna TEP menos provável como hipótese principal.",
        ],
        "comorbidity_weight": 0.35,
    },
    "insuficiência cardíaca descompensada": {
        "positives": ["dispneia progressiva", "ortopneia", "edema de membros inferiores", "fadiga"],
        "negatives": ["sem febre alta", "sem hemoptise", "sem dor torácica pleurítica"],
        "differentials": ["síndrome coronariana aguda", "pneumonia", "tromboembolismo pulmonar"],
        "exams": ["BNP", "radiografia de tórax", "ecocardiograma"],
        "reasoning_for": [
            "Dispneia progressiva com ortopneia e edema de membros inferiores favorece insuficiência cardíaca descompensada.",
        ],
        "reasoning_against": [
            "Sem febre alta ou hemoptise, causas infecciosas ou tromboembólicas agudas ficam menos prováveis como explicação única.",
        ],
        "comorbidity_weight": 0.40,
    },
    "pneumonia": {
        "positives": ["febre alta", "tosse produtiva", "dispneia progressiva", "fadiga"],
        "negatives": ["sem rigidez de nuca", "sem dor abdominal localizada", "sem sinais focais neurológicos"],
        "differentials": ["virose inespecífica", "insuficiência cardíaca descompensada", "sepse"],
        "exams": ["radiografia de tórax", "hemograma", "proteína C reativa"],
        "reasoning_for": [
            "Febre alta com tosse produtiva e dispneia progressiva favorece pneumonia.",
        ],
        "reasoning_against": [
            "A ausência de sinais meníngeos e abdominais focais reduz outras síndromes agudas não respiratórias.",
        ],
        "comorbidity_weight": 0.25,
    },
    "meningite bacteriana": {
        "positives": ["febre alta", "rigidez de nuca", "cefaleia intensa", "confusão mental"],
        "negatives": ["sem foco respiratório claro", "sem padrão típico de sinusite isolada"],
        "differentials": ["encefalite", "hemorragia subaracnoide", "sinusite aguda complicada"],
        "exams": ["punção lombar", "hemograma", "hemocultura"],
        "reasoning_for": [
            "Febre alta, rigidez de nuca, cefaleia intensa e confusão mental favorecem meningite bacteriana.",
        ],
        "reasoning_against": [
            "A presença de sinais meníngeos fortes supera diagnósticos mais benignos de cefaleia ou vias aéreas superiores.",
        ],
        "comorbidity_weight": 0.20,
    },
    "hemorragia subaracnoide": {
        "positives": ["cefaleia súbita intensa", "vômitos", "fotofobia", "rebaixamento do nível de consciência"],
        "negatives": ["sem síndrome viral típica", "sem evolução arrastada"],
        "differentials": ["enxaqueca", "meningite bacteriana", "AVC"],
        "exams": ["tomografia de crânio", "punção lombar se necessário", "angioTC"],
        "reasoning_for": [
            "Cefaleia súbita de forte intensidade com vômitos e alteração neurológica favorece hemorragia subaracnoide.",
        ],
        "reasoning_against": [
            "A apresentação abrupta torna cefaleias primárias menos prováveis como primeira hipótese.",
        ],
        "comorbidity_weight": 0.15,
    },
    "tromboembolismo pulmonar": {
        "positives": ["dispneia súbita", "dor torácica pleurítica", "hemoptise", "taquicardia"],
        "negatives": ["sem tosse produtiva importante", "sem padrão típico de refluxo"],
        "differentials": ["síndrome coronariana aguda", "pneumonia", "ansiedade"],
        "exams": ["d-dímero", "angioTC pulmonar", "gasometria arterial"],
        "reasoning_for": [
            "Dispneia súbita associada a dor pleurítica e hemoptise favorece tromboembolismo pulmonar.",
        ],
        "reasoning_against": [
            "A ausência de tosse produtiva relevante reduz pneumonia como hipótese principal.",
        ],
        "comorbidity_weight": 0.15,
    },
}

OUT_OF_SCOPE_SPECIALTIES = [
    ("dermatologia especializada", "Lesão cutânea crônica com necessidade de dermatoscopia e foto detalhada."),
    ("psiquiatria especializada", "Ajuste fino de esquema antidepressivo em transtorno psiquiátrico complexo."),
    ("oftalmologia especializada", "Queixa visual com necessidade de fundoscopia e avaliação oftalmológica específica."),
    ("oncologia especializada", "Discussão de conduta oncológica sistêmica e esquema terapêutico especializado."),
]


def pick_comorbidities(weight):
    if random.random() > weight:
        return []
    size = 1 if random.random() < 0.7 else 2
    return random.sample(COMORBIDADES_POOL, size)


def format_patient_context(age, sex_abbr, comorbidades, symptom_lines):
    context = f"Contexto do paciente:\nPaciente {sex_abbr}, {age} anos."
    if comorbidades:
        context += f"\nHistórico: {', '.join(comorbidades)}"
    else:
        context += "\nHistórico: não informado. Comorbidades registradas: nenhuma."
    context += "\n\nSintomas relatados:\n" + "; ".join(symptom_lines)
    return context


def answer_block(status, summary, hypothesis, differentials, exams, reasoning, missing_data=None, specialty=None):
    parts = [
        f"Status da análise:\n{status}",
        f"\nResumo clínico:\n{summary}",
        f"\nHipótese diagnóstica principal:\n{hypothesis}",
        "\nDiagnósticos diferenciais:\n" + "".join(f"- {d}\n" for d in differentials),
        "\nExames recomendados:\n" + "".join(f"- {e}\n" for e in exams),
        f"\nRaciocínio clínico:\n{reasoning}",
    ]
    if missing_data:
        parts.append("\nDados faltantes:\n" + "".join(f"- {item}\n" for item in missing_data))
    if specialty:
        parts.append(f"\nEspecialidade sugerida:\n{specialty}")
    return "".join(parts).strip()


def generate_supported_case():
    disease = random.choice(list(SUPPORTED_CASES.keys()))
    cfg = SUPPORTED_CASES[disease]
    age = random.randint(18, 85)
    sex = random.choice(SEXOS)
    sex_abbr = 'M' if sex == 'masculino' else 'F'
    comorbidades = pick_comorbidities(cfg.get('comorbidity_weight', 0.2))

    positives = random.sample(cfg['positives'], k=min(len(cfg['positives']), random.randint(2, min(4, len(cfg['positives'])))))
    negatives = random.sample(cfg['negatives'], k=min(len(cfg['negatives']), random.randint(1, min(3, len(cfg['negatives'])))))
    symptom_lines = positives + negatives
    random.shuffle(symptom_lines)

    summary = f"Paciente apresentando {'; '.join(positives)}."
    if comorbidades:
        summary += f" Histórico relevante: {', '.join(comorbidades)}."
    reasoning_parts = []
    if comorbidades:
        reasoning_parts.append(f"Comorbidades como {', '.join(comorbidades)} devem ser consideradas na estratificação de risco.")
    reasoning_parts.extend(random.sample(cfg['reasoning_for'], k=1))
    reasoning_parts.extend(random.sample(cfg['reasoning_against'], k=1))
    reasoning = ' '.join(reasoning_parts)

    context = format_patient_context(age, sex_abbr, comorbidades, symptom_lines)
    answer = answer_block(
        status='supported_hypothesis',
        summary=summary,
        hypothesis=disease,
        differentials=cfg['differentials'],
        exams=cfg['exams'],
        reasoning=reasoning,
    )
    return context, random.choice(QUESTIONS), answer, disease, 'supported_hypothesis'


def generate_insufficient_data_case():
    patterns = [
        ["dor de cabeça leve"],
        ["dor abdominal vaga"],
        ["mal-estar inespecífico"],
        ["tontura inespecífica"],
        ["dor torácica sem descrição de padrão"],
    ]
    missing_templates = {
        "dor de cabeça leve": ["tempo de evolução", "presença de febre", "déficit neurológico", "fotofobia ou vômitos"],
        "dor abdominal vaga": ["localização da dor", "tempo de evolução", "presença de náuseas/vômitos/diarreia", "sinais peritoneais"],
        "mal-estar inespecífico": ["sintomas associados", "temperatura aferida", "tempo de evolução", "foco principal da queixa"],
        "tontura inespecífica": ["tipo de tontura", "gatilhos", "déficit neurológico", "sinais cardiovasculares"],
        "dor torácica sem descrição de padrão": ["caráter da dor", "relação com esforço", "dispneia", "irradiação e duração"],
    }
    symptom_lines = random.choice(patterns)
    base = symptom_lines[0]
    age = random.randint(18, 85)
    sex = random.choice(SEXOS)
    sex_abbr = 'M' if sex == 'masculino' else 'F'
    comorbidades = []
    context = format_patient_context(age, sex_abbr, comorbidades, symptom_lines)
    summary = f"Paciente com queixa inespecífica de {base}, sem dados discriminativos suficientes para priorização segura."
    reasoning = "Os dados disponíveis são insuficientes para sustentar uma hipótese principal com segurança. É necessário esclarecer características-chave da queixa antes de priorizar causas graves ou benignas."
    answer = answer_block(
        status='insufficient_data',
        summary=summary,
        hypothesis='indeterminada com os dados atuais',
        differentials=['necessita refinamento clínico'],
        exams=['coleta complementar de história e exame físico dirigido'],
        reasoning=reasoning,
        missing_data=missing_templates[base],
    )
    return context, random.choice(QUESTIONS), answer, 'indeterminada com os dados atuais', 'insufficient_data'


def generate_out_of_scope_case():
    specialty, description = random.choice(OUT_OF_SCOPE_SPECIALTIES)
    age = random.randint(18, 85)
    sex = random.choice(SEXOS)
    sex_abbr = 'M' if sex == 'masculino' else 'F'
    context = format_patient_context(age, sex_abbr, [], [description])
    summary = "Caso demanda avaliação especializada e não deve receber diagnóstico conclusivo pelo assistente generalista sem dados apropriados."
    reasoning = "A pergunta exige exame específico, contexto especializado ou decisão terapêutica de subespecialidade. O comportamento seguro é reconhecer limitação de escopo e orientar encaminhamento."
    answer = answer_block(
        status='out_of_scope',
        summary=summary,
        hypothesis='fora do escopo principal do assistente',
        differentials=['avaliação especializada necessária'],
        exams=['encaminhamento para avaliação especializada'],
        reasoning=reasoning,
        specialty=specialty,
    )
    return context, random.choice(QUESTIONS), answer, 'fora do escopo principal do assistente', 'out_of_scope'


def generate_dataset(total_examples=6000):
    dataset = []
    labels = []
    statuses = []
    for _ in range(total_examples):
        roll = random.random()
        if roll < 0.68:
            context, question, answer, label, status = generate_supported_case()
        elif roll < 0.86:
            context, question, answer, label, status = generate_insufficient_data_case()
        else:
            context, question, answer, label, status = generate_out_of_scope_case()

        dataset.append({
            "messages": [
                {"role": "system", "content": "Você é um assistente clínico orientado à segurança. Use apenas os dados fornecidos, reconheça insuficiência de dados quando apropriado e declare fora de escopo quando necessário."},
                {"role": "user", "content": context + "\n\nPergunta:\n" + question},
                {"role": "assistant", "content": answer},
            ]
        })
        labels.append(label)
        statuses.append(status)
    return dataset, labels, statuses


def extract_main_hypothesis(answer_text):
    marker = "Hipótese diagnóstica principal:"
    if marker not in answer_text:
        return None
    tail = answer_text.split(marker, 1)[1].strip()
    return tail.splitlines()[0].strip() if tail else None


def validate_dataset(dataset):
    hypotheses = []
    statuses = []
    for row in dataset:
        answer = row['messages'][-1]['content']
        hypothesis = extract_main_hypothesis(answer)
        if hypothesis:
            hypotheses.append(hypothesis)
        if 'Status da análise:' in answer:
            status = answer.split('Status da análise:', 1)[1].strip().splitlines()[0].strip()
            statuses.append(status)
    return Counter(hypotheses), Counter(statuses)


TOTAL_EXAMPLES = 6000
dataset, label_counts_input, status_counts_input = generate_dataset(TOTAL_EXAMPLES)

with open('dataset_clinical_cases_generated.jsonl', 'w', encoding='utf8') as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

label_counter, status_counter = validate_dataset(dataset)
print(f'Dataset gerado: {len(dataset)} casos')
print('\nDistribuição de status:')
for key, value in status_counter.most_common():
    print(f'- {key}: {value}')
print('\nHipóteses mais frequentes:')
for key, value in label_counter.most_common(15):
    print(f'- {key}: {value}')
print('\nArquivo salvo em: dataset_clinical_cases_generated.jsonl')


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

pasta = '/content/drive/MyDrive/FIAP-TC-Fase3'
os.makedirs(pasta, exist_ok=True)

!cp 'dataset_clinical_cases_generated.jsonl' '{pasta}/'
print(f'Salvo em: {pasta}/dataset_clinical_cases_generated.jsonl')
